# Trabajo Práctico N° 1: Evaluación y Análisis de Redes Neuronales CBOW
**Asignatura**: Aprendizaje Automático Avanzado (UNAHUR)
Este notebook se destina **exclusivamente a la visualización y análisis de resultados** utilizando los modelos previamente entrenados y resguardados en formato .

## Sección 1: Carga de un Modelo Entrenado (.npz) y Gráfico de Pérdida por Época
Carga la estructura completa del modelo desde un archivo  y grafica la función de pérdida.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Asegurar acceso a los módulos del directorio codigo/
ruta_raiz = Path.cwd()
if str(ruta_raiz) not in sys.path:
    sys.path.append(str(ruta_raiz))

from codigo.modelo_cbow_cupy import cargar_modelo

# Cargar el modelo resguardado en formato .npz
ruta_modelo = "respaldos/modelo_cbow_w5_epoca_10.npz"
modelo = cargar_modelo(ruta_modelo)

# Graficar curva de pérdida por época
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
epocas = range(1, len(modelo["historial_perdida"]) + 1)
plt.plot(epocas, modelo["historial_perdida"], marker="o", color="#2b5c8f", linewidth=2.5, label="Pérdida CBOW")
plt.title("Evolución de la Pérdida de Entrenamiento por Época", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio (Cross-Entropy)", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## Sección 2: Búsqueda de Similaridad de Palabras sobre la Matriz $
Evalúa palabras similares calculando el producto interno o la similaridad de coseno directamente sobre las filas de la matriz $ y el arreglo .

In [ ]:
def buscar_palabras_similares(
    palabra_buscada: str,
    W: np.ndarray,
    vocabulario_palabras: np.ndarray,
    top_k: int = 10,
    tipo_similaridad: str = "producto_interno"
) -> list[tuple[str, float]]:
    """
    Busca las palabras más similares a una palabra objetivo dada basándose en las filas de la matriz W.
    
    :param palabra_buscada: Palabra objetivo ingresada en texto.
    :param W: Matriz de pesos de entrada (|V| x N).
    :param vocabulario_palabras: Arreglo de cadenas de texto con las |V| palabras del vocabulario.
    :param top_k: Cantidad de vecinos más cercanos a retornar.
    :param tipo_similaridad: 'producto_interno' o 'coseno'.
    :return: Lista de tuplas (palabra, puntaje).
    """
    palabra_normalizada = palabra_buscada.lower().strip()
    vocab_lista = list(vocabulario_palabras)
    
    if palabra_normalizada not in vocab_lista:
        print(f"Advertencia: La palabra '{palabra_buscada}' no se encuentra en el vocabulario.")
        palabra_normalizada = "<UNK>"

    indice_objetivo = vocab_lista.index(palabra_normalizada)

    if hasattr(W, "get"):
        matriz_W = W.get()
    else:
        matriz_W = np.asarray(W)

    vector_palabra = matriz_W[indice_objetivo, :]

    if tipo_similaridad == "producto_interno":
        puntajes = np.dot(matriz_W, vector_palabra)
    elif tipo_similaridad == "coseno":
        norma_vector = np.linalg.norm(vector_palabra)
        normas_matriz = np.linalg.norm(matriz_W, axis=1)
        producto_punto = np.dot(matriz_W, vector_palabra)
        puntajes = producto_punto / (normas_matriz * norma_vector + 1e-12)
    else:
        raise ValueError(f"Tipo de similaridad no soportado: '{tipo_similaridad}'")

    indices_ordenados = np.argsort(puntajes)[::-1]

    resultados = []
    for idx in indices_ordenados:
        if idx == indice_objetivo:
            continue
        resultados.append((vocabulario_palabras[idx], float(puntajes[idx])))
        if len(resultados) >= top_k:
            break

    return resultados

# Probar la búsqueda de similaridad sobre el modelo cargado
palabra_test = "sol"
similares = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="producto_interno"
)

print(f"Palabras más similares a '{palabra_test}':")
for pal, score in similares:
    print(f" - {pal:15s}: {score:.4f}")


## Sección 3: Comparación de Curvas de Pérdida entre Dos Modelos o Checkpoints (.npz)
Carga dos modelos entrenados en formato  y grafica de forma superpuesta sus curvas de pérdida.

In [ ]:
# Cargar dos modelos resguardados para comparativa
ruta_modelo_1 = "respaldos/modelo_cbow_w5_epoca_10.npz"
ruta_modelo_2 = "respaldos/modelo_cbow_w5_epoca_10.npz"  # Reemplazar con la ruta del segundo modelo

modelo_1 = cargar_modelo(ruta_modelo_1)
modelo_2 = cargar_modelo(ruta_modelo_2)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(modelo_1["historial_perdida"]) + 1), modelo_1["historial_perdida"], label="Modelo 1", color="#2b5c8f", linewidth=2)
plt.plot(range(1, len(modelo_2["historial_perdida"]) + 1), modelo_2["historial_perdida"], label="Modelo 2", color="#d95f02", linestyle="--", linewidth=2)
plt.title("Comparativa de Curvas de Pérdida entre Dos Modelos CBOW", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()
